# 01 · Data, features and baseline
Football injury risk prediction on SoccerMon (Zenodo record 10033832).

Run top to bottom in Colab. Section 2 downloads the data; section 3 is where you map the real column names once you've seen them.

## 1. Setup

In [ ]:
!git clone https://github.com/<your-username>/football-injury-risk.git 2>/dev/null || true
%cd football-injury-risk
!pip install -q -r requirements.txt

In [ ]:
import sys, os, json, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "src")
from features import complete_calendar, add_load_features, FEATURES
os.makedirs("data", exist_ok=True)

## 2. Download the dataset
The Zenodo API lists the files in the record. Check sizes before downloading, since the GPS part is huge and you don't need it for this project.

In [ ]:
rec = requests.get("https://zenodo.org/api/records/10033832", timeout=60).json()
for f in rec["files"]:
    print(f"{f['key']:<60} {f['size']/1e6:10.1f} MB")

In [ ]:
# Download only the small subjective files (training load, wellness, injury).
# Set WANTED after reading the listing above.
WANTED = ["subjective.zip"]
for f in rec["files"]:
    if f["key"] in WANTED:
        url = f["links"]["self"]
        out = os.path.join("data", f["key"])
        with requests.get(url, stream=True, timeout=120) as r, open(out, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
        print("saved", out)

In [ ]:
!cd data && unzip -oq subjective.zip && find subjective -type f | sort

## 3. Build the daily table
`daily_load.csv` is wide (one column per player), `injury.csv` is long. Players file a report on every day an injury is active, so consecutive daily reports count as one injury. A report is a new onset when the player had none in the previous 7 days.

The pre-computed `acwr`, `atl`, `ctl`, `monotony` and `strain` files are not used. The pipeline recomputes them from daily load so the windows are known.

In [ ]:
from data import build_daily
base = "data/subjective"
daily = build_daily(f"{base}/training-load/daily_load.csv", f"{base}/injury/injury.csv", gap_days=7)
print(daily.shape, "players:", daily.player_id.nunique(), "| onsets:", int(daily.injury.sum()))
print(daily.date.min().date(), "->", daily.date.max().date())
daily.head()

Check how many onsets there are per month and per team before choosing split dates. With only a few hundred events, the split matters a lot.

In [ ]:
d = daily[daily.injury == 1].copy()
d["team"] = d.player_id.str.split("-").str[0]
print(d.groupby([d.date.dt.to_period("M"), "team"]).size().unstack(fill_value=0))
print("\nonsets per player:", d.groupby("player_id").size().describe().round(1).to_dict())

### Reporting check
Injury onsets drop to zero from April to August 2021, while 2020 has several every month. Before trusting any label, check whether players kept logging load in those months. If they did, the missing injuries are probably missing reports, and every day in that window is a false negative.

In [ ]:
inj_raw = pd.read_csv(f"{base}/injury/injury.csv")
inj_raw["date"] = pd.to_datetime(inj_raw["timestamp"], format="%d.%m.%Y")
m = pd.DataFrame({
    "injury_reports": inj_raw.groupby(inj_raw.date.dt.to_period("M")).size(),
    "onsets": d.groupby(d.date.dt.to_period("M")).size(),
    "load_days": daily[daily.load > 0].groupby(daily.date.dt.to_period("M")).size(),
    "active_players": daily[daily.load > 0].groupby(daily.date.dt.to_period("M")).player_id.nunique(),
}).fillna(0).astype(int)
print(m)
print("\ntotal injury report rows:", len(inj_raw), "| players with any report:", inj_raw.player_name.nunique())
print(inj_raw["type"].str.extract(r':"(\w+)"')[0].value_counts().to_dict())

## 4. Features and target
Target = any injury onset in the next 7 days. Rolling features use past days only.

In [ ]:
df = add_load_features(complete_calendar(daily), horizon=7)
df = df.dropna(subset=["target", "acute_7d", "chronic_28d"]).copy()
df["target"] = df["target"].astype(int)
print(df.shape, "positive rate:", round(df.target.mean(), 4))

## 5. Split
Split by time, not by random row. Adjacent days are nearly identical, so a random split leaks information and inflates every metric. Defaults below: train on 2020, validate on the first part of 2021, test on the rest. Change the dates if the onset table above shows a very uneven spread.

In [ ]:
cut1, cut2 = pd.Timestamp("2021-01-01"), pd.Timestamp("2021-07-01")
train, val, test = df[df.date < cut1], df[(df.date >= cut1) & (df.date < cut2)], df[df.date >= cut2]
for n, p in [("train", train), ("val", val), ("test", test)]:
    print(f"{n:5} rows={len(p):6} positives={int(p.target.sum()):4} rate={p.target.mean():.4f}")
X = lambda d: d[FEATURES]; y = lambda d: d["target"]

## 6. Baselines
Each model has to beat the one before it. PR-AUC is the main metric, since accuracy is meaningless when about 1 to 3% of rows are positive. A model that guesses at random scores a PR-AUC equal to the positive rate.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def report(name, yt, p):
    p = np.asarray(p, dtype=float)
    out = f"{name:22} ROC-AUC={roc_auc_score(yt, p):.3f}  PR-AUC={average_precision_score(yt, p):.3f}"
    if 0 <= p.min() and p.max() <= 1:          # Brier only makes sense for probabilities
        out += f"  Brier={brier_score_loss(yt, p):.4f}"
    print(out)

yv = y(val)
report("prevalence (no skill)", yv, np.full(len(yv), y(train).mean()))
report("ACWR only", yv, val["acwr"].fillna(val["acwr"].median()))   # higher ACWR = higher risk

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   LogisticRegression(class_weight="balanced", max_iter=1000))
lr.fit(X(train), y(train))
report("logistic regression", yv, lr.predict_proba(X(val))[:, 1])

## Next
- LightGBM + SHAP in `02_model.ipynb`
- Add wellness features (sleep, fatigue, soreness, readiness are in `subjective/wellness/`)
- Calibration plot and alert threshold (precision at a fixed number of alerts per team per week)
- Per-player error analysis. Check that results don't come from a handful of injury-prone players.